In [28]:
%matplotlib widget

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))


from joblib import Parallel, delayed, parallel_config

from evaluation.evaluate_experiment import ExperimentPlotter
from evaluation.run_agent import get_actions
from serialization.serialize_experiment import load_experiment, read_experiment_metadata


In [ ]:
EXPERIMENTS = {
    "2026-11-09T19-23-37-941649Z" : {"name": "Full (Gamma = 0.993, En/Ergo)", "plot": True},
    "2026-11-16T12-51-57-456763Z" : {"name": "Diagonal (Gamma = 0.993, En/Ergo)", "plot": True},

    "2026-11-18T10-49-22-464396Z" : {"name": "Full (Gamma = 1.000, En/Ergo)", "plot": True},
    "2026-11-18T14-53-36-457913Z" : {"name": "Diagonal (Gamma = 1.000, En/Ergo)", "plot": True},

    "2026-11-16T19-33-13-715702Z" : {"name": "Full (Gamma = 0.993, Ergo)", "plot": True},
    "2026-11-16T20-54-35-535476Z" : {"name": "Diagonal (Gamma = 0.993, Ergo)", "plot": True},

    "2026-11-16T16-08-54-483021Z" : {"name": "Full (Gamma = 1.000, Ergo)", "plot": True},
    "2026-11-16T17-39-58-100875Z" : {"name": "Diagonal (Gamma = 1.000, Ergo)", "plot": True},

    "2026-11-16T11-27-51-031470Z" : {"name": "Full Reduced (Gamma = 0.993, En/Ergo)", "plot": False},
    "2026-11-16T14-16-44-692193Z" : {"name": "Diagonal Reduced (Gamma = 0.993, En/Ergo)", "plot": False},
    
    "2026-11-18T13-10-16-038142Z":  {"name": "Diagonal [FAIL] (Gamma = 1.000, En/Ergo)", "plot": False},

}

EXPERIMENT_IDs = list(EXPERIMENTS.keys())

In [3]:
experiments = {}

for EXPERIMENT_ID in EXPERIMENT_IDs:

    metadata = read_experiment_metadata(EXPERIMENT_ID)

    ENV_PARAMS = metadata["env_params"]
    HYPER_PARAMS = metadata["hyper_params"]

    seed = metadata["seed"]

    agent, gym_env, _ = load_experiment(EXPERIMENT_ID)

    env = gym_env.qm_env

    experiments[EXPERIMENT_ID] = {
        "agent": agent,
        "gym_env": gym_env, 
        "env": env
    }

In [4]:
def replay(agent, gym_env, env):

    _, actions, _ = get_actions(
        gym_env, 
        agent, 
        seed = seed, 
        deterministic=False
    )

    return env.replay(actions)

experiment_histories = {}

for i, EXPERIMENT_ID in enumerate(experiments):

    histories = []

    agent = experiments[EXPERIMENT_ID]["agent"]
    gym_env = experiments[EXPERIMENT_ID]["gym_env"]
    env = experiments[EXPERIMENT_ID]["env"]

    with parallel_config(
        backend="loky",
        n_jobs=14,
        inner_max_num_threads=1
    ):
        histories = Parallel()(
            delayed(replay)(agent, gym_env, env)
            for _ in range(50)
        )

    experiment_histories[EXPERIMENT_ID] = histories

    print(
        "[" + str(i+1) + "/" + str(len(EXPERIMENTS)) + "] " 
        + EXPERIMENTS[EXPERIMENT_ID]["name"] 
        + " (" + EXPERIMENT_ID + ")" 
        + " complete!"
    )

[1/11] Full (Gamma:1, En/Ergo) (2026-11-18T10-49-22-464396Z) complete!
[2/11] Diagonal (Gamma:1, En/Ergo) (2026-11-18T14-53-36-457913Z) complete!
[3/11] Diagonal [FAIL] (Gamma:1, En/Ergo) (2026-11-18T13-10-16-038142Z) complete!
[4/11] Full (Gamma:0.993, Ergo) (2026-11-16T19-33-13-715702Z) complete!
[5/11] Diagonal (Gamma:0.993, Ergo) (2026-11-16T20-54-35-535476Z) complete!
[6/11] Full (Gamma:1, Ergo) (2026-11-16T16-08-54-483021Z) complete!
[7/11] Diagonal (Gamma:1, Ergo) (2026-11-16T17-39-58-100875Z) complete!
[8/11] Full (Gamma:0.993, En/Ergo) (2026-11-09T19-23-37-941649Z) complete!
[9/11] Diagonal (Gamma:0.993, En/Ergo) (2026-11-16T12-51-57-456763Z) complete!
[10/11] Reduced Full (Gamma:0.993, En/Ergo) (2026-11-16T11-27-51-031470Z) complete!
[11/11] Reduced Diagonal (Gamma:0.993, En/Ergo) (2026-11-16T14-16-44-692193Z) complete!


In [29]:
plotter = ExperimentPlotter(
    EXPERIMENTS,
    experiment_histories,
    max_cols=3,
    fig_width=5,
    fig_height=3.5,
)
plotter.show()